# 🔧 Interaktiver CO₂-Booster Parameter-Explorer

**Live-Simulation** der transkritischen CO₂-Booster-Kälteanlage mit Ecalia Scroll-Expander.

Nutze die Schieberegler, um die Hauptparameter anzupassen — das log(p)-h-Diagramm und die Kennwerte (COP, Leistung, Massenstrom) aktualisieren sich sofort.

> **Basis:** BITZER Software v7.1.2 Auslegungsdaten von Kälte Fischer GmbH (7 Betriebspunkte, NK-Stufe)

In [25]:
%matplotlib widget

import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import matplotlib.pyplot as plt
from IPython.display import display
from CoolProp.CoolProp import PropsSI as PSI

from ipywidgets import (
    FloatSlider, ToggleButtons, VBox, HBox, Output,
    Button, Layout, HTML as HTMLWidget, Label,
)

from src.co2_booster import CO2BoosterFlashgasIWT, CO2BoosterParallel
from src.operating_points import (
    OP_SUMMER_NO_PV, OP_SUMMER_WITH_PV,
    OP_TRANSITION_NO_PV, OP_TRANSITION_WITH_PV,
    OP_WRG_NO_PV, OP_WRG_WITH_PV,
    OP_WINTER, OperatingPoint,
)
from src.plotting import (
    _draw_logph_background, _draw_model_on_axis, _annotate_canonical,
    ECALIA_BLUE, ECALIA_RED, ECALIA_ORANGE,
)
from fluprodia import FluidPropertyDiagram
from copy import deepcopy


plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 120,
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 9,
})


## Betriebspunkte & Referenzwerte (BITZER)

Die 4 validierten Betriebspunkte von Kälte Fischer / BITZER als Presets:

In [26]:
# ── Operating point presets ──
# Each preset has both a "without PV" and optionally a "with PV" operating point.
PRESETS = {
    "Hochsommer": {
        "op": OP_SUMMER_NO_PV,
        "op_pv": OP_SUMMER_WITH_PV,
        "cop_bitzer": 1.56,
        "cop_bitzer_pv": 1.85,
        "eta_s": 0.691,
        "mode": "transcritical",
    },
    "Übergang": {
        "op": OP_TRANSITION_NO_PV,
        "op_pv": OP_TRANSITION_WITH_PV,
        "cop_bitzer": 2.97,
        "cop_bitzer_pv": 3.43,
        "eta_s": 0.684,
        "mode": "subcritical",
    },
    "WRG-Modus": {
        "op": OP_WRG_NO_PV,
        "op_pv": OP_WRG_WITH_PV,
        "cop_bitzer": 2.36,
        "cop_bitzer_pv": 2.48,
        "eta_s": 0.679,
        "mode": "transcritical",
    },
    "Winter": {
        "op": OP_WINTER,
        "op_pv": None,  # PV cannot operate in winter
        "cop_bitzer": 4.54,
        "cop_bitzer_pv": None,
        "eta_s": 0.655,
        "mode": "subcritical",
    },
}

# Medium pressure default
P_MEDIUM_DEFAULT = 38.0

# Pre-compute the fluprodia diagram once (expensive)
diagram = FluidPropertyDiagram("CO2")
diagram.set_unit_system(T="°C", p="bar", h="kJ/kg")

print("✅ Presets & diagram ready")

✅ Presets & diagram ready


## Interaktive Steuerung

Passe die Parameter über die Slider an. Das log(p)-h-Diagramm und die Kennwerte aktualisieren sich nach Klick auf **▶ Berechnen**.

**Tipp:** Nutze die Preset-Buttons um schnell zu einem BITZER-Betriebspunkt zu springen.

In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# Slider definitions
# ═══════════════════════════════════════════════════════════════════════
slider_layout = Layout(width='380px')
label_layout = Layout(width='180px')

# --- Cycle parameters ---
sl_p_high = FloatSlider(value=93.7, min=70, max=120, step=0.5,
    description='p_HP [bar]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

sl_T_gc_out = FloatSlider(value=38.0, min=15, max=50, step=0.5,
    description='T_GC,out [°C]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

sl_T_evap = FloatSlider(value=-10.0, min=-15, max=0, step=0.5,
    description='T_evap [°C]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

sl_superheat = FloatSlider(value=6.0, min=2, max=12, step=0.5,
    description='ΔT_SH [K]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

sl_superheat_sl = FloatSlider(value=4.0, min=0, max=10, step=0.5,
    description='ΔT_SL [K]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

sl_Q0 = FloatSlider(value=98.0, min=50, max=150, step=1.0,
    description='Q₀_NK [kW]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.0f')

sl_p_medium = FloatSlider(value=38.0, min=30, max=50, step=0.5,
    description='p_medium [bar]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

# --- Efficiency parameters ---
sl_eta_s = FloatSlider(value=0.685, min=0.50, max=0.85, step=0.005,
    description='η_s Kompr.', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.3f')

sl_eta_exp = FloatSlider(value=0.70, min=0.30, max=0.90, step=0.01,
    description='η_s Expander', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.2f')

# --- Mode selection ---
toggle_mode = ToggleButtons(
    options=['Ventil (Baseline)', 'HD-Expander', 'MD-Expander', 'HD+MD Dual'],
    value='Ventil (Baseline)',
    description='Expansion:',
    style={'description_width': '80px', 'button_width': '130px'},
)

toggle_cycle_mode = ToggleButtons(
    options=['transcritical', 'subcritical'],
    value='transcritical',
    description='Modus:',
    style={'description_width': '80px', 'button_width': '130px'},
)

toggle_pv = ToggleButtons(
    options=['Ohne PV', 'Mit PV (Parallelverdichter)'],
    value='Ohne PV',
    description='PV:',
    style={'description_width': '80px', 'button_width': '170px'},
)

toggle_iwt = ToggleButtons(
    options=['IWT nach Exp.', 'IWT vor Exp.'],
    value='IWT nach Exp.',
    description='IWT:',
    style={'description_width': '80px', 'button_width': '130px'},
)

sl_T_iwt_sh = FloatSlider(value=10.0, min=2, max=20, step=0.5,
    description='IWT Überhitzung [K]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

sl_eta_pv = FloatSlider(value=0.685, min=0.50, max=0.85, step=0.005,
    description='η_s PV-Kompr.', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.3f')

sl_T_cond = FloatSlider(value=27.0, min=10, max=40, step=0.5,
    description='T_cond [°C]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

sl_subcooling = FloatSlider(value=3.0, min=0, max=10, step=0.5,
    description='Unterkühlung [K]', style={'description_width': '120px'},
    layout=slider_layout, readout_format='.1f')

# ═══════════════════════════════════════════════════════════════════════
# Preset buttons
# ═══════════════════════════════════════════════════════════════════════
preset_buttons = {}
for name in PRESETS:
    btn = Button(description=name, layout=Layout(width='auto'),
                 button_style='info')
    preset_buttons[name] = btn

btn_compute = Button(description='▶ Berechnen', button_style='success',
                     layout=Layout(width='200px', height='40px'),
                     icon='play')

# ═══════════════════════════════════════════════════════════════════════
# Output areas
# ═══════════════════════════════════════════════════════════════════════
plot_output = Output(layout=Layout(width='100%'))
kpi_output = HTMLWidget(value='<i>Noch keine Berechnung durchgeführt.</i>',
                        layout=Layout(width='100%', min_height='120px'))
status_output = HTMLWidget(value='', layout=Layout(width='100%'))

# Track currently loaded preset
_active_preset = {'name': None}

# ═══════════════════════════════════════════════════════════════════════
# Core computation and plotting
# ═══════════════════════════════════════════════════════════════════════

def run_simulation():
    """Run the TESPy simulation with current slider values and update plot + KPIs."""
    mode = toggle_cycle_mode.value
    exp_choice = toggle_mode.value
    use_pv = toggle_pv.value.startswith('Mit')
    iwt_pos = "before_exp" if toggle_iwt.value.startswith('IWT vor') else "after_exp"

    expansion_hp = "expander" if exp_choice in ('HD-Expander', 'HD+MD Dual') else "valve"
    expansion_fg = "expander" if exp_choice in ('MD-Expander', 'HD+MD Dual') else "valve"

    # PV model doesn't support MD-Expander (PV replaces flash gas bypass)
    if use_pv and expansion_fg == "expander":
        expansion_fg = "valve"

    # Build operating point from slider values
    op = OperatingPoint(
        name="Interaktiv",
        mode=mode,
        has_parallel_compressor=use_pv,
        T_evap=sl_T_evap.value,
        superheat_evap=sl_superheat.value,
        superheat_suction=sl_superheat_sl.value,
        Q0_NK=sl_Q0.value,
        p_high=sl_p_high.value if mode == "transcritical" else None,
        T_gc_out=sl_T_gc_out.value if mode == "transcritical" else None,
        T_cond=sl_T_cond.value if mode == "subcritical" else None,
        subcooling=sl_subcooling.value if mode == "subcritical" else None,
        p_medium=sl_p_medium.value,
        p_medium_pv=sl_p_medium.value if use_pv else None,
        T_iwt_superheat_pv=sl_T_iwt_sh.value if use_pv else None,
    )

    status_output.value = '<span style="color:#888;">⏳ Berechnung läuft...</span>'

    # ── Solve current configuration ──
    try:
        if use_pv:
            model = CO2BoosterParallel(
                expansion_device_hp=expansion_hp,
                parallel_compressor_efficiency=sl_eta_pv.value,
                compressor_efficiency=sl_eta_s.value,
                expander_efficiency=sl_eta_exp.value,
            )
        else:
            model = CO2BoosterFlashgasIWT(
                expansion_device_hp=expansion_hp,
                expansion_device_fg=expansion_fg,
                iwt_position=iwt_pos,
                compressor_efficiency=sl_eta_s.value,
                expander_efficiency=sl_eta_exp.value,
            )
        model.setup_network(iterinfo=False)
        model.set_boundary_conditions(op=op)
        model.solve()
    except Exception as e:
        status_output.value = f'<span style="color:red;">❌ Fehler: {e}</span>'
        return

    if not model.converged:
        status_output.value = '<span style="color:red;">❌ Nicht konvergiert! Parameter prüfen.</span>'
        return

    # ── Also solve baseline (valve+valve) for comparison ──
    baseline_cop = None
    try:
        if use_pv:
            bl = CO2BoosterParallel(
                expansion_device_hp="valve",
                parallel_compressor_efficiency=sl_eta_pv.value,
                compressor_efficiency=sl_eta_s.value,
                expander_efficiency=sl_eta_exp.value,
            )
        else:
            bl = CO2BoosterFlashgasIWT(
                expansion_device_hp="valve", expansion_device_fg="valve",
                iwt_position=iwt_pos,
                compressor_efficiency=sl_eta_s.value,
                expander_efficiency=sl_eta_exp.value,
            )
        bl.setup_network(iterinfo=False)
        bl.set_boundary_conditions(op=op)
        bl.solve()
        if bl.converged:
            baseline_cop = bl.calculate_cop_cooling()
    except Exception:
        pass

    # ── Extract results ──
    cop = model.calculate_cop_cooling()
    power = model.get_power_breakdown()
    P_comp = sum(d['P_kW'] for d in power.values() if d['type'] == 'compressor')
    P_exp = sum(d['P_kW'] for d in power.values() if d['type'] == 'expander')
    P_net = P_comp + P_exp
    states = model.get_state_points()

    # Mass flow from first connection
    m_dot = next(iter(states.values()))['m']

    # Discharge temperature
    comp_out_key = 'nk_compressor-gas_cooler'
    T_discharge = states[comp_out_key]['T'] if comp_out_key in states else float('nan')

    # Pressure ratio
    p_evap = PSI("P", "Q", 1, "T", 273.15 + sl_T_evap.value, "CO2") / 1e5
    if mode == "transcritical":
        pi = sl_p_high.value / p_evap
    else:
        p_cond = PSI("P", "Q", 0, "T", 273.15 + sl_T_cond.value, "CO2") / 1e5
        pi = p_cond / p_evap

    # Flash gas ratio
    drum_liq_key = 'flash_drum-mpev'
    drum_vap_keys = ['flash_drum-fgbv', 'flash_drum-iwt_flashgas']  # IWT or PV topology
    m_liq = states.get(drum_liq_key, {}).get('m', 0)
    m_vap = 0
    for k in drum_vap_keys:
        if k in states:
            m_vap = states[k].get('m', 0)
            break
    x_flash = m_vap / (m_liq + m_vap) if (m_liq + m_vap) > 0 else 0

    # ── Determine dynamic x-axis range from state points ──
    h_values = [s['h'] for s in states.values() if 'h' in s]
    h_max_data = max(h_values) if h_values else 500
    x_max_plot = max(575, h_max_data + 25)  # at least 575, or data + 25 kJ/kg margin

    # ── Update plot ──
    with plot_output:
        plot_output.clear_output(wait=True)
        fig, ax = plt.subplots(1, figsize=(10, 6))

        _draw_logph_background(diagram, fig, ax,
                               x_min=100, x_max=x_max_plot, y_min=15, y_max=130)

        color = ECALIA_ORANGE if expansion_hp == "expander" or expansion_fg == "expander" else ECALIA_BLUE
        _draw_model_on_axis(model, diagram, ax, color=color, linewidth=2.2,
                            label=exp_choice)

        _annotate_canonical(model, ax, fontsize=8)

        title = f"log(p)-h  |  {exp_choice}"
        if use_pv:
            title += " + PV"
        title += f"  |  COP = {cop:.3f}"
        if _active_preset['name']:
            title += f"  [{_active_preset['name']}]"
        ax.set_title(title, fontweight='bold', fontsize=12)
        ax.set_xlabel("Enthalpie h [kJ/kg]")
        ax.set_ylabel("Druck p [bar]")
        ax.legend(loc='upper right', fontsize=8)

        fig.tight_layout()
        plt.show()

    # ── Update KPI panel ──
    T_dis_color = 'red' if T_discharge > 140 else ('orange' if T_discharge > 120 else 'inherit')

    # BITZER reference
    bitzer_ref = ""
    if _active_preset['name'] and _active_preset['name'] in PRESETS:
        preset_info = PRESETS[_active_preset['name']]
        if use_pv and preset_info.get('cop_bitzer_pv'):
            cop_b = preset_info['cop_bitzer_pv']
            bitzer_ref = f"<tr><td><b>COP (BITZER Ref. mit PV)</b></td><td><b>{cop_b:.2f}</b></td></tr>"
        else:
            cop_b = preset_info['cop_bitzer']
            bitzer_ref = f"<tr><td><b>COP (BITZER Referenz)</b></td><td><b>{cop_b:.2f}</b></td></tr>"

    # Baseline comparison
    bl_row = ""
    if baseline_cop is not None and (expansion_hp == "expander" or expansion_fg == "expander"):
        delta = (cop - baseline_cop) / baseline_cop * 100
        bl_row = (
            f"<tr><td>COP Baseline (V+V)</td><td>{baseline_cop:.3f}</td></tr>"
            f'<tr><td><b>ΔCOP vs. Baseline</b></td>'
            f'<td style="color:{"green" if delta > 0 else "red"};"><b>{"+" if delta > 0 else ""}{delta:.1f} %</b></td></tr>'
        )

    # Expander breakdown
    exp_rows = ""
    if expansion_hp == "expander" or expansion_fg == "expander":
        for comp_name, d in power.items():
            if d['type'] == 'expander':
                label = "HD-Expander" if "hpev" in comp_name else "MD-Expander"
                exp_rows += f"<tr><td>  {label}</td><td>{d['P_kW']:.2f} kW</td></tr>"

    kpi_html = f"""
    <table style="font-size:13px; border-collapse:collapse; width:100%;">
    <tr style="background:#f0f0f0;"><th style="text-align:left; padding:4px 8px;">Kennwert</th>
        <th style="text-align:left; padding:4px 8px;">Wert</th></tr>
    <tr><td style="padding:2px 8px;"><b>COP (Kühlung)</b></td>
        <td style="padding:2px 8px;"><b style="font-size:16px;">{cop:.3f}</b></td></tr>
    {bitzer_ref}
    {bl_row}
    <tr><td style="padding:2px 8px;">P_Kompressor</td><td style="padding:2px 8px;">{P_comp:.2f} kW</td></tr>
    <tr><td style="padding:2px 8px;">P_Expander</td><td style="padding:2px 8px;">{P_exp:.2f} kW</td></tr>
    {exp_rows}
    <tr><td style="padding:2px 8px;"><b>P_netto</b></td><td style="padding:2px 8px;"><b>{P_net:.2f} kW</b></td></tr>
    <tr><td style="padding:2px 8px;">Massenstrom ṁ</td><td style="padding:2px 8px;">{m_dot:.3f} kg/s</td></tr>
    <tr><td style="padding:2px 8px;">Druckverhältnis π</td><td style="padding:2px 8px;">{pi:.2f}</td></tr>
    <tr><td style="padding:2px 8px;">Flash-Gas Anteil x</td><td style="padding:2px 8px;">{x_flash:.1%}</td></tr>
    <tr><td style="padding:2px 8px;">T_Austritt Kompressor</td>
        <td style="padding:2px 8px; color:{T_dis_color};">{T_discharge:.1f} °C</td></tr>
    </table>
    """
    kpi_output.value = kpi_html
    status_output.value = '<span style="color:green;">✅ Berechnung abgeschlossen</span>'


# ═══════════════════════════════════════════════════════════════════════
# Preset button callbacks
# ═══════════════════════════════════════════════════════════════════════

def make_preset_callback(name):
    def on_click(b):
        preset = PRESETS[name]
        use_pv = toggle_pv.value.startswith('Mit')
        _active_preset['name'] = name

        # Select correct OP based on PV toggle
        if use_pv and preset.get('op_pv') is not None:
            op = preset['op_pv']
        else:
            op = preset['op']
            # If PV requested but not available, switch toggle
            if use_pv and preset.get('op_pv') is None:
                toggle_pv.value = 'Ohne PV'

        # Set cycle mode first
        toggle_cycle_mode.value = preset['mode']

        # Set sliders
        sl_T_evap.value = op.T_evap
        sl_superheat.value = op.superheat_evap
        sl_superheat_sl.value = op.superheat_suction
        sl_Q0.value = op.Q0_NK
        sl_p_medium.value = op.p_medium_pv if (use_pv and op.p_medium_pv) else op.p_medium
        sl_eta_s.value = preset['eta_s']

        if op.T_iwt_superheat_pv is not None:
            sl_T_iwt_sh.value = op.T_iwt_superheat_pv

        if preset['mode'] == 'transcritical':
            sl_p_high.value = op.p_high
            sl_T_gc_out.value = op.T_gc_out
        else:
            sl_T_cond.value = op.T_cond
            sl_subcooling.value = op.subcooling or 3.0

        # Highlight active button
        for n, btn in preset_buttons.items():
            btn.button_style = 'warning' if n == name else 'info'

        run_simulation()
    return on_click

for name, btn in preset_buttons.items():
    btn.on_click(make_preset_callback(name))

btn_compute.on_click(lambda b: run_simulation())

# ═══════════════════════════════════════════════════════════════════════
# Layout
# ═══════════════════════════════════════════════════════════════════════

# Transcritical sliders
tc_box = VBox([sl_p_high, sl_T_gc_out], layout=Layout(display='flex'))

# Subcritical sliders
sc_box = VBox([sl_T_cond, sl_subcooling], layout=Layout(display='none'))

# PV-specific sliders (hidden by default)
pv_box = VBox([sl_eta_pv, sl_T_iwt_sh], layout=Layout(display='none'))

def on_mode_change(change):
    if change['new'] == 'transcritical':
        tc_box.layout.display = 'flex'
        sc_box.layout.display = 'none'
    else:
        tc_box.layout.display = 'none'
        sc_box.layout.display = 'flex'

toggle_cycle_mode.observe(on_mode_change, names='value')

def on_pv_change(change):
    use_pv = change['new'].startswith('Mit')
    pv_box.layout.display = 'flex' if use_pv else 'none'
    # MD-Expander not available with PV — update toggle options
    if use_pv:
        toggle_mode.options = ['Ventil (Baseline)', 'HD-Expander']
        if toggle_mode.value in ('MD-Expander', 'HD+MD Dual'):
            toggle_mode.value = 'Ventil (Baseline)'
    else:
        toggle_mode.options = ['Ventil (Baseline)', 'HD-Expander', 'MD-Expander', 'HD+MD Dual']

toggle_pv.observe(on_pv_change, names='value')

left_col = VBox([
    Label('── Kreislauf ──'),
    toggle_cycle_mode,
    tc_box,
    sc_box,
    sl_T_evap, sl_superheat, sl_superheat_sl,
    sl_Q0, sl_p_medium,
])

right_col = VBox([
    Label('── Konfiguration ──'),
    toggle_pv,
    pv_box,
    toggle_iwt,
    toggle_mode,
    Label('── Effizienz ──'),
    sl_eta_s, sl_eta_exp,
    Label(''),
    Label('── Presets (BITZER) ──'),
    HBox(list(preset_buttons.values())),
    Label(''),
    btn_compute,
])

controls = HBox([left_col, right_col],
                layout=Layout(justify_content='space-between'))

display(VBox([
    controls,
    status_output,
    plot_output,
    kpi_output,
]))


## Schnellvergleich: Alle Konfigurationen

Klicke unten, um **alle 4 Expander-Konfigurationen** mit den aktuellen Slider-Werten gleichzeitig zu berechnen und als Balkendiagramm zu vergleichen.

In [ ]:
compare_output = Output(layout=Layout(width='100%'))
compare_kpi = HTMLWidget(value='', layout=Layout(width='100%'))
btn_compare = Button(description='▶ Alle Konfigurationen vergleichen',
                     button_style='primary', layout=Layout(width='300px'),
                     icon='bar-chart')

def run_comparison(b):
    """Compute all configurations with current slider values and compare."""
    mode = toggle_cycle_mode.value
    use_pv = toggle_pv.value.startswith('Mit')
    iwt_pos = "before_exp" if toggle_iwt.value.startswith('IWT vor') else "after_exp"

    op = OperatingPoint(
        name="Vergleich",
        mode=mode,
        has_parallel_compressor=use_pv,
        T_evap=sl_T_evap.value,
        superheat_evap=sl_superheat.value,
        superheat_suction=sl_superheat_sl.value,
        Q0_NK=sl_Q0.value,
        p_high=sl_p_high.value if mode == "transcritical" else None,
        T_gc_out=sl_T_gc_out.value if mode == "transcritical" else None,
        T_cond=sl_T_cond.value if mode == "subcritical" else None,
        subcooling=sl_subcooling.value if mode == "subcritical" else None,
        p_medium=sl_p_medium.value,
        p_medium_pv=sl_p_medium.value if use_pv else None,
        T_iwt_superheat_pv=sl_T_iwt_sh.value if use_pv else None,
    )

    if use_pv:
        # PV: only Baseline and HD-Expander (MD not applicable)
        configs = [
            ("PV Baseline (V)", "valve", "valve", ECALIA_BLUE),
            ("PV + HD-Exp",     "expander", "valve", ECALIA_ORANGE),
        ]
    else:
        configs = [
            ("Baseline (V+V)", "valve", "valve", ECALIA_BLUE),
            ("MD-Expander",    "valve", "expander", ECALIA_RED),
            ("HD-Expander",    "expander", "valve", ECALIA_ORANGE),
            ("HD+MD Dual",     "expander", "expander", "#2ca02c"),
        ]

    results = {}
    models = {}
    for label, hp, fg, color in configs:
        try:
            if use_pv:
                m = CO2BoosterParallel(
                    expansion_device_hp=hp,
                    parallel_compressor_efficiency=sl_eta_pv.value,
                    compressor_efficiency=sl_eta_s.value,
                    expander_efficiency=sl_eta_exp.value,
                )
            else:
                m = CO2BoosterFlashgasIWT(
                    expansion_device_hp=hp, expansion_device_fg=fg,
                    iwt_position=iwt_pos,
                    compressor_efficiency=sl_eta_s.value,
                    expander_efficiency=sl_eta_exp.value,
                )
            m.setup_network(iterinfo=False)
            m.set_boundary_conditions(op=op)
            m.solve()
            if m.converged:
                cop = m.calculate_cop_cooling()
                pwr = m.get_power_breakdown()
                P_c = sum(d['P_kW'] for d in pwr.values() if d['type'] == 'compressor')
                P_e = sum(d['P_kW'] for d in pwr.values() if d['type'] == 'expander')
                results[label] = {"cop": cop, "P_comp": P_c, "P_exp": P_e,
                                  "P_net": P_c + P_e, "color": color}
                models[label] = m
            else:
                results[label] = None
        except Exception:
            results[label] = None

    # ── Bar chart ──
    with compare_output:
        compare_output.clear_output(wait=True)

        fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

        valid = {k: v for k, v in results.items() if v is not None}
        names = list(valid.keys())
        colors = [valid[n]['color'] for n in names]

        # COP
        cops = [valid[n]['cop'] for n in names]
        axes[0].barh(names, cops, color=colors)
        axes[0].set_xlabel('COP')
        axes[0].set_title('COP Vergleich')
        for i, v in enumerate(cops):
            axes[0].text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=9)
        axes[0].set_xlim(right=max(cops) * 1.25 if cops else 1)

        # Net power
        pnets = [valid[n]['P_net'] for n in names]
        axes[1].barh(names, pnets, color=colors)
        axes[1].set_xlabel('P_netto [kW]')
        axes[1].set_title('Netto-Leistungsaufnahme')
        for i, v in enumerate(pnets):
            axes[1].text(v + 0.2, i, f'{v:.1f}', va='center', fontsize=9)
        axes[1].set_xlim(right=max(pnets) * 1.2 if pnets else 1)

        # Expander recovery (show as positive for consistent bar direction)
        pexps = [abs(valid[n]['P_exp']) for n in names]
        axes[2].barh(names, pexps, color=colors)
        axes[2].set_xlabel('|P_Expander| [kW]')
        axes[2].set_title('Expander-Rückgewinnung')
        for i, v in enumerate(pexps):
            axes[2].text(v + 0.1, i, f'{v:.1f}', va='center', fontsize=9)
        axes[2].set_xlim(right=max(pexps) * 1.25 if pexps else 1)

        pv_label = " + PV" if use_pv else ""
        iwt_label = f" | IWT {toggle_iwt.value.split(' ')[-1]}"
        fig.suptitle(f'Konfigurationsvergleich — {mode}{pv_label}{iwt_label}', fontweight='bold')
        fig.tight_layout(rect=(0, 0, 1, 0.93), w_pad=3.5)
        plt.show()

    # ── KPI table ──
    bl_cop = results.get("Baseline (V+V)", {})
    bl_cop_val = bl_cop['cop'] if bl_cop else None

    rows = ""
    for label, r in results.items():
        if r is None:
            rows += f"<tr><td>{label}</td><td colspan=4 style='color:red;'>Nicht konvergiert</td></tr>"
            continue
        delta = ""
        if bl_cop_val and label != "Baseline (V+V)":
            d = (r['cop'] - bl_cop_val) / bl_cop_val * 100
            delta = f'<span style="color:{"green" if d > 0 else "red"};">{"+" if d > 0 else ""}{d:.1f}%</span>'
        rows += (
            f"<tr><td><b>{label}</b></td>"
            f"<td>{r['cop']:.3f} {delta}</td>"
            f"<td>{r['P_comp']:.1f}</td>"
            f"<td>{r['P_exp']:.1f}</td>"
            f"<td>{r['P_net']:.1f}</td></tr>"
        )

    compare_kpi.value = f"""
    <table style="font-size:12px; border-collapse:collapse; width:100%; margin-top:8px;">
    <tr style="background:#e0e0e0;">
        <th style="padding:4px 8px; text-align:left;">Konfiguration</th>
        <th style="padding:4px 8px;">COP</th>
        <th style="padding:4px 8px;">P_Kompr. [kW]</th>
        <th style="padding:4px 8px;">P_Exp. [kW]</th>
        <th style="padding:4px 8px;">P_netto [kW]</th>
    </tr>
    {rows}
    </table>"""

btn_compare.on_click(run_comparison)

display(VBox([btn_compare, compare_output, compare_kpi]))